# Week 6 — Day 5: CNNs — Convolutional Neural Networks on MNIST

## Goal
Build first CNN on MNIST handwritten digit dataset.
Target: above 98% accuracy on test set.
Result: 99.27% ✅

---

## Key Concepts

### What is Convolution?
A filter (small matrix e.g. 3×3) slides across the image multiplying
its values with image pixels and summing them. Detects features —
edges, corners, curves. Early layers detect simple edges, deeper
layers detect complex shapes.

### What is MaxPooling?
Takes maximum value from each 2×2 region. Shrinks spatial size of
feature map — reduces computation and makes network less sensitive
to exact position of features.
28×28 → MaxPool → 14×14 → MaxPool → 7×7

### Why CNN over nn.Linear for images?
nn.Linear flattens image to 784 numbers — loses all spatial
information (which pixel is next to which). CNN preserves spatial
structure — filter sees neighboring pixels together, detecting
edges and shapes the way humans do.

### CrossEntropyLoss vs BCELoss
- BCELoss — binary classification only (2 classes: survived/died)
- CrossEntropyLoss — multiclass (10 digit classes 0-9)
- CrossEntropyLoss combines Softmax + loss in one step
- Rule: BCELoss for 2 classes, CrossEntropyLoss for 3+ classes

### DataLoader and batches
60,000 images can't fit in memory at once. DataLoader serves
64 images at a time (1 batch). Each epoch = 937 batches = all
60,000 images seen once.

```python
for data, target in train_loader:
    # data shape: [64, 1, 28, 28]
    # target shape: [64] — actual digit labels
```

---

## Data Loading

```python
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True,
                                download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False,
                                download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)
```

- ToTensor() — converts image to tensor, scales 0-255 to 0-1
- Normalize(0.1307, 0.3081) — MNIST mean and std, centers data around 0
- shuffle=True for train — different order each epoch prevents memorization
- shuffle=False for test — consistent evaluation

**Dataset size: 60,000 train, 10,000 test**

---

## CNN Architecture

```python
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 28x28 → 28x28
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 28x28 → 14x14

            nn.Conv2d(32, 64, kernel_size=3, padding=1),  # 14x14 → 14x14
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 14x14 → 7x7
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)  # 10 digit classes
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # flatten 64×7×7 → 3136
        x = self.fc_layers(x)
        return x
```

**Total parameters: 421,834**

### Data flow through network:

# Load MNIST Dataset and Build First CNN

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True,
                                download=True, transform=transform)
test_dataset  = datasets.MNIST(root='./data', train=False,
                                download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples : {len(test_dataset)}")

100.0%
100.0%
100.0%
100.0%

Train samples: 60000
Test samples : 10000


# Write the CNN architecture:

In [4]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv_layers = nn.Sequential(
            # Conv block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 28x28 → 14x14
            
            # Conv block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),  # 14x14 → 7x7
        )
        
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)  # 10 classes (digits 0-9)
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # flatten
        x = self.fc_layers(x)
        return x

model = MnistCNN()
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

MnistCNN(
  (conv_layers): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Linear(in_features=3136, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)

Total parameters: 421,834


# Training Loop:

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_accs = []
test_accs  = []

for epoch in range(10):
    # training phase
    model.train()
    correct = 0
    total   = 0
    
    for data, target in train_loader:
        optimizer.zero_grad()
        output = model(data)
        loss   = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        # count correct predictions
        pred     = output.argmax(dim=1)  # highest score = predicted digit
        correct += (pred == target).sum().item()
        total   += target.size(0)
    
    train_acc = correct / total
    
    # validation phase
    model.eval()
    correct = 0
    total   = 0
    with torch.no_grad():
        for data, target in test_loader:
            output   = model(data)
            pred     = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total   += target.size(0)
    
    test_acc = correct / total
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    
    print(f"Epoch {epoch+1:2d} | "
          f"Train Acc: {train_acc:.4f} | "
          f"Test Acc: {test_acc:.4f}")

print(f"\nBest Test Accuracy: {max(test_accs):.4f}")

Epoch  1 | Train Acc: 0.9248 | Test Acc: 0.9822
Epoch  2 | Train Acc: 0.9667 | Test Acc: 0.9886
Epoch  3 | Train Acc: 0.9747 | Test Acc: 0.9879
Epoch  4 | Train Acc: 0.9768 | Test Acc: 0.9888
Epoch  5 | Train Acc: 0.9804 | Test Acc: 0.9892
Epoch  6 | Train Acc: 0.9820 | Test Acc: 0.9908
Epoch  7 | Train Acc: 0.9849 | Test Acc: 0.9913
Epoch  8 | Train Acc: 0.9855 | Test Acc: 0.9923
Epoch  9 | Train Acc: 0.9868 | Test Acc: 0.9919
Epoch 10 | Train Acc: 0.9881 | Test Acc: 0.9927

Best Test Accuracy: 0.9927
